In [ ]:
# --- Section 0: Setup Project Environment (Kaggle) ---
import os
import sys
import shutil
import subprocess

# ── Step 1: Install lightweight dependencies only ───────────
# IMPORTANT: Do NOT reinstall torch/torchvision on Kaggle unlessจำเป็นจริง ๆ
# เพราะอาจทำให้ CUDA build ไม่ตรงกับ GPU ของเครื่องรัน
pkgs = [
    "transformers>=4.30",
    "matplotlib>=3.7",
    "pillow>=10.0",
    "numpy>=1.24",
    "scikit-learn>=1.3",
]

print("📦 Installing extra dependencies...")
cmd = [sys.executable, "-m", "pip", "install", "-q", "--no-input", *pkgs]
subprocess.run(cmd, check=True)
print("✅ Install complete\n")

# ── Step 2: Copy project from Kaggle input dataset ──────────
# Kaggle แตก zip ให้แล้ว -> ชี้ไปที่โฟลเดอร์ dataset ได้เลย
SOURCE_DIR = "/kaggle/input/datasets/priyadatechasaratool/mae-classification-code01"  # เปลี่ยนเป็น dataset folder จริงของคุณ
WORK_DIR = "/kaggle/working/project"

if not os.path.exists(SOURCE_DIR):
    raise FileNotFoundError(
        f"SOURCE_DIR not found: {SOURCE_DIR}. "
        "Please update SOURCE_DIR to your Kaggle dataset folder."
    )

if not os.path.exists(WORK_DIR):
    print(f"📂 Copying files from\n  {SOURCE_DIR}\n→ {WORK_DIR} ...")
    shutil.copytree(SOURCE_DIR, WORK_DIR)
    print("✅ Copy complete!\n")
else:
    print(f"✅ Already exists: {WORK_DIR}\n")

os.chdir(WORK_DIR)
if WORK_DIR not in sys.path:
    sys.path.insert(0, WORK_DIR)

print(f"📍 CWD: {os.getcwd()}")
print("📦 sys.path updated")
print("ℹ️ HF Hub warning about unauthenticated requests is normal if HF_TOKEN is not set.\n")

# ── Step 3: Verify key files ───────────────────────────────
key_files = [
    "start_implementation.ipynb",
    "data/animals10.py",
    "models/unet.py",
    "training/mae_trainer.py",
    "training/classification.py",
    "training/evaluation.py",
    "training/unet.py",
    "utils/common.py",
    "pyproject.toml",
]

print("🔍 Verifying key files...")
all_ok = True
for f in key_files:
    exists = os.path.exists(f)
    status = "✅" if exists else "❌ MISSING"
    print(f"  {status}  {f}")
    if not exists:
        all_ok = False

if all_ok:
    print("\n✅ All files OK — ready to run training!")
else:
    print("\n⚠️ Some files are missing. Check your dataset upload.")

In [ ]:
# --- Section 1: Import Libraries and Configure Environment ---
from dataclasses import dataclass
from pathlib import Path
import random

import matplotlib.pyplot as plt
import numpy as np
import torch

from utils.common import set_seed


@dataclass
class Config:
    DATA_ROOT: str = "/kaggle/input/datasets/alessiocorrado99/animals10/raw-img"
    CKPT_DIR: str = "/kaggle/working/checkpoints"
    RESULT_DIR: str = "/kaggle/working/results"
    BATCH_SIZE: int = 32
    EPOCHS_MAE: int = 50
    EPOCHS_UNET: int = 50
    EPOCHS_CLS: int = 20
    LR: float = 1e-4
    MASK_RATIO: float = 0.75
    IMG_SIZE: int = 224
    SEED: int = 42
    NUM_WORKERS: int = 2
    CLS_TRAIN_MODE: str = "partial"  # "partial" or "end_to_end"
    UNFREEZE_LAST_BLOCKS: int = 1
    UNFREEZE_VIT_LAYERNORM: bool = True
    USE_WEIGHTED_SAMPLER: bool = True


cfg = Config()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# CUDA sanity check: บางครั้ง torch build ไม่ตรงกับ GPU runtime บน Kaggle
if device.type == "cuda":
    try:
        _ = torch.zeros(1, device=device) + 1
        torch.cuda.synchronize()
    except Exception as exc:
        print(f"⚠️ CUDA unavailable at runtime ({exc}). Fallback to CPU.")
        device = torch.device("cpu")

set_seed(cfg.SEED)

Path(cfg.CKPT_DIR).mkdir(parents=True, exist_ok=True)
Path(cfg.RESULT_DIR).mkdir(parents=True, exist_ok=True)

print(f"Device: {device}")
print(f"Dataset root: {cfg.DATA_ROOT}")
print(f"Checkpoint dir: {cfg.CKPT_DIR}")
print(f"Result dir: {cfg.RESULT_DIR}")
print(f"Classifier train mode: {cfg.CLS_TRAIN_MODE}")
print(f"Use weighted sampler: {cfg.USE_WEIGHTED_SAMPLER}")

In [ ]:
# --- Section 2: Load and Inspect the Dataset ---
from collections import Counter
from PIL import Image

from data.animals10 import build_split, discover_class_directories

# Animals-10 folder names (Italian) -> English labels
it_to_en = {
    "cane": "dog",
    "cavallo": "horse",
    "elefante": "elephant",
    "farfalla": "butterfly",
    "gallina": "chicken",
    "gatto": "cat",
    "mucca": "cow",
    "pecora": "sheep",
    "ragno": "spider",
    "scoiattolo": "squirrel",
}
expected_english_classes = {
    "dog", "horse", "elephant", "butterfly", "chicken",
    "cat", "cow", "sheep", "spider", "squirrel",
}

class_dirs = discover_class_directories(cfg.DATA_ROOT)
split = build_split(cfg.DATA_ROOT, val_fraction=0.2, seed=cfg.SEED)
idx_to_class = {index: class_name for class_name, index in split.class_to_idx.items()}

dataset_classes_it = set(split.class_to_idx.keys())
unknown_it = sorted(dataset_classes_it - set(it_to_en.keys()))
missing_it = sorted(set(it_to_en.keys()) - dataset_classes_it)
if unknown_it or missing_it:
    raise ValueError(
        f"Unexpected class folders. unknown={unknown_it}, missing={missing_it}. "
        "Please check DATA_ROOT points to Animals-10 raw-img."
    )

# Classification labels in English
class_to_idx_en = {it_to_en[class_name]: index for class_name, index in split.class_to_idx.items()}
idx_to_class_en = {index: english for english, index in class_to_idx_en.items()}

if set(class_to_idx_en.keys()) != expected_english_classes:
    raise ValueError("English class mapping is incomplete or incorrect.")

print(f"Classes found: {len(class_dirs)}")
print("Original folder names (IT):", sorted(split.class_to_idx.keys()))
print("Classification labels (EN):", [idx_to_class_en[i] for i in sorted(idx_to_class_en.keys())])
print(f"Train samples: {len(split.train_samples)}")
print(f"Val samples: {len(split.val_samples)}")

train_counts = Counter(label for _, label in split.train_samples)
val_counts = Counter(label for _, label in split.val_samples)
print("Train distribution:", {idx_to_class_en[idx]: count for idx, count in train_counts.items()})
print("Val distribution:", {idx_to_class_en[idx]: count for idx, count in val_counts.items()})

sample_paths = [path for path, _ in split.train_samples[:4]]
fig, axes = plt.subplots(1, len(sample_paths), figsize=(16, 4))
for axis, sample_path in zip(axes, sample_paths):
    with Image.open(sample_path) as image:
        axis.imshow(image.convert("RGB"))
    axis.set_title(Path(sample_path).parent.name)
    axis.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# --- Section 3: Define Image Preprocessing and Augmentation ---
from torchvision import transforms as T
from data.animals10 import IMAGENET_MEAN, IMAGENET_STD

train_transform = T.Compose(
    [
        T.RandomResizedCrop(
            cfg.IMG_SIZE,
            scale=(0.2, 1.0),
            interpolation=T.InterpolationMode.BICUBIC,
        ),
        T.RandomHorizontalFlip(p=0.5),
        T.ToTensor(),
        T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ]
)

val_transform = T.Compose(
    [
        T.Resize(256, interpolation=T.InterpolationMode.BICUBIC),
        T.CenterCrop(cfg.IMG_SIZE),
        T.ToTensor(),
        T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ]
)

print(train_transform)
print(val_transform)

In [ ]:
# --- Section 4: Build Data Loaders ---
from data.animals10 import build_dataloaders

train_loader, val_loader, split = build_dataloaders(
    root=cfg.DATA_ROOT,
    batch_size=cfg.BATCH_SIZE,
    image_size=cfg.IMG_SIZE,
    val_fraction=0.2,
    seed=cfg.SEED,
    num_workers=cfg.NUM_WORKERS,
    train_transform=train_transform,
    val_transform=val_transform,
    use_weighted_sampler=cfg.USE_WEIGHTED_SAMPLER,
)

idx_to_class = {index: class_name for class_name, index in split.class_to_idx.items()}
print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")
print(f"Train sampler: {type(train_loader.sampler).__name__}")
first_batch = next(iter(train_loader))
print(f"First batch image shape: {tuple(first_batch[0].shape)}")
print(f"First batch label shape: {tuple(first_batch[1].shape)}")

In [ ]:
# --- Section 5: Implement the MAE Encoder-Decoder Model ---
from training.mae_trainer import load_mae_model, load_mae_processor, reconstruct_image, reconstruct_mae_images

mae_processor = load_mae_processor()
mae_model = load_mae_model(mask_ratio=cfg.MASK_RATIO).to(device)
mae_model.eval()

sample_images = first_batch[0][:2].to(device)
with torch.no_grad():
    try:
        mae_outputs = mae_model(pixel_values=sample_images)
        mae_loss = mae_outputs.loss.item()
    except RuntimeError as exc:
        error_text = str(exc).lower()
        if "no kernel image is available" in error_text or "cuda error" in error_text:
            print("⚠️ CUDA kernel compatibility issue detected. Switching to CPU runtime.")
            device = torch.device("cpu")
            mae_model = mae_model.to(device)
            sample_images = sample_images.to(device)
            mae_outputs = mae_model(pixel_values=sample_images)
            mae_loss = mae_outputs.loss.item()
        else:
            raise

print(f"MAE loss on first batch: {mae_loss:.6f}")
print("MAE model ready for continual pre-training and reconstruction.")

In [ ]:
# --- Section 6: Add the Classification Head ---
from transformers import ViTForImageClassification
from training.classification import load_mae_encoder_weights_into_classifier, set_classifier_train_mode

label2id = class_to_idx_en
id2label = {idx: label for idx, label in idx_to_class_en.items()}

classification_model = ViTForImageClassification.from_pretrained(
    "facebook/vit-mae-base",
    num_labels=len(split.class_to_idx),
    ignore_mismatched_sizes=True,
    label2id=label2id,
    id2label=id2label,
).to(device)

classification_model = load_mae_encoder_weights_into_classifier(
    classification_model,
    mae_checkpoint_path=None,
)

trainable_params = set_classifier_train_mode(
    classification_model,
    mode=cfg.CLS_TRAIN_MODE,
    unfreeze_last_blocks=cfg.UNFREEZE_LAST_BLOCKS,
    unfreeze_vit_layernorm=cfg.UNFREEZE_VIT_LAYERNORM,
)

print("Classification head attached.")
print(classification_model.__class__.__name__)
print("Model labels (EN):", classification_model.config.id2label)
print(f"Trainable parameters: {trainable_params:,}")

In [ ]:
# --- Section 7: Define Loss Functions and Optimizer ---
import torch.nn.functional as F

from models.unet import UNet
from utils.common import create_grad_scaler
from training.unet import apply_patch_mask

unet_model = UNet().to(device)
mae_optimizer = torch.optim.AdamW(mae_model.parameters(), lr=cfg.LR)
unet_optimizer = torch.optim.AdamW(unet_model.parameters(), lr=cfg.LR)
# Optimize only parameters marked as trainable by CLS_TRAIN_MODE
cls_optimizer = torch.optim.AdamW((p for p in classification_model.parameters() if p.requires_grad), lr=cfg.LR)
mae_scaler = create_grad_scaler(device)
unet_scaler = create_grad_scaler(device)
cls_scaler = create_grad_scaler(device)

reconstruction_loss = torch.nn.MSELoss()
classification_loss = torch.nn.CrossEntropyLoss()

print("Optimizers and loss functions are ready.")

In [ ]:
# --- Section 8: Train the Model (REAL TRAINING LOOP) ---
import torch
from training.mae_trainer import MAETrainer
from training.unet import UNetReconstructionTrainer
from training.classification import ViTClassifierTrainer

# ตั้งค่า Early Stopping กลางสำหรับทุกโมเดล
EARLY_STOP_PATIENCE = 5 

# =====================================================================
# 1. 🚀 เทรน MAE ให้รู้จักโครงสร้างภาพ
# =====================================================================
print(f"--- Starting MAE Pre-training ({cfg.EPOCHS_MAE} Epochs) ---", flush=True)
mae_trainer = MAETrainer(mae_model, mae_optimizer, device, mask_ratio=cfg.MASK_RATIO, scaler=mae_scaler)

best_val_loss_mae = float('inf')
patience_counter_mae = 0

for epoch in range(cfg.EPOCHS_MAE):
    mae_train_loss = mae_trainer.train_epoch(train_loader)
    mae_val_loss = mae_trainer.evaluate_epoch(val_loader)
    current_epoch = epoch + 1
    
    print(
        f"MAE Epoch {current_epoch}/{cfg.EPOCHS_MAE} | "
        f"Train Loss: {mae_train_loss:.4f} | Val Loss: {mae_val_loss:.4f}",
        flush=True
    )

    # --- Save Checkpoint ทุกๆ 10 Epoch ---
    if current_epoch % 10 == 0:
        checkpoint_path = f"{cfg.CKPT_DIR}/mae_epoch_{current_epoch}.pt"
        torch.save(mae_model.state_dict(), checkpoint_path)
        print(f"💾 Checkpoint saved: {checkpoint_path}", flush=True)

    # --- Save Best Model & Early Stopping ---
    if mae_val_loss < best_val_loss_mae:
        best_val_loss_mae = mae_val_loss
        patience_counter_mae = 0
        torch.save(mae_model.state_dict(), f"{cfg.CKPT_DIR}/mae_best.pt")
        print(f"🌟 New best MAE saved with Val Loss: {best_val_loss_mae:.4f}", flush=True)
    else:
        patience_counter_mae += 1
        
    if patience_counter_mae >= EARLY_STOP_PATIENCE:
        print(f"🛑 MAE Early stopping triggered at Epoch {current_epoch}!")
        break

# เทรนเสร็จแล้ว เซฟสมอง MAE ตัวสุดท้ายเก็บไว้
torch.save(mae_model.state_dict(), f"{cfg.CKPT_DIR}/mae_final.pt")


# =====================================================================
# 2. 🚀 เทรน U-Net ให้ซ่อมภาพแข่งกับ MAE
# =====================================================================
print(f"\n--- Starting U-Net Training ({cfg.EPOCHS_UNET} Epochs) ---", flush=True)
unet_trainer = UNetReconstructionTrainer(unet_model, unet_optimizer, device, mask_ratio=cfg.MASK_RATIO, scaler=unet_scaler)

best_val_loss_unet = float('inf')
patience_counter_unet = 0

for epoch in range(cfg.EPOCHS_UNET):
    unet_train_loss = unet_trainer.train_epoch(train_loader)
    unet_val_loss = unet_trainer.evaluate_epoch(val_loader)
    current_epoch = epoch + 1
    
    print(
        f"U-Net Epoch {current_epoch}/{cfg.EPOCHS_UNET} | "
        f"Train Loss: {unet_train_loss:.4f} | Val Loss: {unet_val_loss:.4f}"
        , flush=True
    )

    # --- Save Checkpoint ทุกๆ 10 Epoch ---
    if current_epoch % 10 == 0:
        checkpoint_path = f"{cfg.CKPT_DIR}/unet_epoch_{current_epoch}.pt"
        torch.save(unet_model.state_dict(), checkpoint_path)
        print(f"💾 Checkpoint saved: {checkpoint_path}", flush=True)

    # --- Save Best Model & Early Stopping ---
    if unet_val_loss < best_val_loss_unet:
        best_val_loss_unet = unet_val_loss
        patience_counter_unet = 0
        torch.save(unet_model.state_dict(), f"{cfg.CKPT_DIR}/unet_best.pt")
        print(f"🌟 New best U-Net saved with Val Loss: {best_val_loss_unet:.4f}", flush=True)
    else:
        patience_counter_unet += 1
        
    if patience_counter_unet >= EARLY_STOP_PATIENCE:
        print(f"🛑 U-Net Early stopping triggered at Epoch {current_epoch}!", flush=True)
        break

# เทรนเสร็จแล้ว เซฟสมอง U-Net ตัวสุดท้ายเก็บไว้
torch.save(unet_model.state_dict(), f"{cfg.CKPT_DIR}/unet_final.pt")


# =====================================================================
# 3. 🚀 เทรน Classification ทายชื่อสัตว์
# =====================================================================
print(f"\n--- Starting Classification Training ({cfg.EPOCHS_CLS} Epochs) ---", flush=True)
cls_trainer = ViTClassifierTrainer(classification_model, cls_optimizer, device, scaler=cls_scaler)

best_val_acc_cls = 0.0 # Classification ใช้ Accuracy วัดความเก่ง (ยิ่งมากยิ่งดี)
patience_counter_cls = 0

for epoch in range(cfg.EPOCHS_CLS):
    cls_train_loss, cls_train_acc = cls_trainer.train_epoch(train_loader)
    cls_val_loss, cls_val_acc = cls_trainer.evaluate_epoch(val_loader)
    current_epoch = epoch + 1
    
    print(
        f"CLS Epoch {current_epoch}/{cfg.EPOCHS_CLS} | "
        f"Train Acc: {cls_train_acc:.4f} | Val Acc: {cls_val_acc:.4f} | Val Loss: {cls_val_loss:.4f}"
        , flush=True
    )

    # --- Save Checkpoint ทุกๆ 10 Epoch ---
    if current_epoch % 10 == 0:
        checkpoint_path = f"{cfg.CKPT_DIR}/cls_epoch_{current_epoch}.pt"
        torch.save(classification_model.state_dict(), checkpoint_path)
        print(f"💾 Checkpoint saved: {checkpoint_path}", flush=True)

    # --- Save Best Model & Early Stopping (ใช้ Val Acc ในการตัดสิน) ---
    if cls_val_acc > best_val_acc_cls:
        best_val_acc_cls = cls_val_acc
        patience_counter_cls = 0
        torch.save(classification_model.state_dict(), f"{cfg.CKPT_DIR}/cls_best.pt")
        print(f"🌟 New best Classification model saved with Val Acc: {best_val_acc_cls:.4f}", flush=True)
    else:
        patience_counter_cls += 1
        
    if patience_counter_cls >= EARLY_STOP_PATIENCE:
        print(f"🛑 Classification Early stopping triggered at Epoch {current_epoch}!", flush=True)
        break

# เทรนเสร็จแล้ว เซฟสมอง Classification ตัวสุดท้ายเก็บไว้
torch.save(classification_model.state_dict(), f"{cfg.CKPT_DIR}/cls_final.pt")
print("\n🎉 Training Complete! All models saved.")

In [ ]:
# --- Section 9: Evaluate Reconstruction and Classification Metrics ---
from training.evaluation import compare_reconstruction_on_batch, evaluate_classifier

comparison_batch = next(iter(val_loader))
comparison_image_path = Path(cfg.RESULT_DIR) / "comparison" / "sample_comparison.png"
reconstruction_metrics = compare_reconstruction_on_batch(
    mae_model=mae_model,
    unet_model=unet_model,
    batch=comparison_batch,
    device=device,
    mask_ratio=cfg.MASK_RATIO,
    output_path=comparison_image_path,
)
classification_metrics = evaluate_classifier(classification_model, val_loader, device)

print("Reconstruction metrics:", reconstruction_metrics)
print("Classification metrics:", classification_metrics)

In [ ]:
# --- Section 10: Save Model Checkpoints and Inference Outputs ---
from utils.common import save_json

summary_payload = {
    "device": str(device),
    "data_root": cfg.DATA_ROOT,
    "class_labels_en": [idx_to_class_en[i] for i in sorted(idx_to_class_en.keys())],
    "mae_val_mse": float(mae_val_loss),
    "unet_val_mse": float(unet_val_loss),
    "classification_val_loss": float(cls_val_loss),
    "classification_val_accuracy": float(cls_val_acc),
    "comparison_image_path": str(comparison_image_path),
}

save_json(Path(cfg.RESULT_DIR) / "run_summary.json", summary_payload)
print("Saved summary to:", Path(cfg.RESULT_DIR) / "run_summary.json")
print("Comparison image saved to:", comparison_image_path)
print("Checkpoint directories:", cfg.CKPT_DIR)